# How Harness² actually works

This notebook calls the Harness² Python API directly. We create an initial harness, run one task through three recursive rounds, and inspect every proposed, probed, and composed harness.

Run it in **Linux or WSL2** from `harness2-main`. No GPU or model credentials are needed.

## The algorithm in 12 lines

```python
current_harness = initial_harness
base_result = execute(task, current_harness)

for t in range(1, k + 1):
    invariant_edit = propose(current_harness, task, base_result, dimension='INV')
    domain_edit    = propose(current_harness, task, base_result, dimension='DOM')
    inv_result = execute(task, invariant_edit)
    dom_result = execute(task, domain_edit)
    composed = compose(current_harness, inv_result, dom_result, history)
    composed_result = execute(task, composed)
    current_harness = composed  # recursion happens here
```

The real code also records evidence, validates harnesses, selects a final harness, and stores experience for later tasks.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
assert (ROOT / 'pyproject.toml').is_file(), 'Launch Jupyter from harness2-main.'
if os.name == 'nt':
    raise RuntimeError('Open this repository in WSL2/Linux; native Windows lacks fcntl.')

from harness2.demo import DemoAdapter, DemoProposer
from harness2.core.schedule import run_task
print('Repository:', ROOT)

## 1. Create the task and initial harness

`DemoAdapter` creates ledger tasks and a base harness. The task asks for every signed amount, but the initial script incorrectly keeps only positive values.

In [ ]:
OUTPUT = ROOT / 'runs' / 'direct_api_tutorial'
adapter = DemoAdapter(OUTPUT)
task_id = 'ledger-1'

print('TASK:', adapter.task_prompt(task_id))
print('\nINPUT:')
print((adapter.task_files(task_id) / 'ledger.csv').read_text())
print('INITIAL SYSTEM PROMPT:')
print((adapter.base_harness() / 'systemprompt.md').read_text())
print('INITIAL SCRIPT:')
print((adapter.base_harness() / 'scripts/summarize.py').read_text())

The correct total is `100.25 - 20.10 + 5.00 = 85.15`. First call the executor directly with the initial harness.

In [ ]:
base_run = adapter.execute(task_id, adapter.base_harness(), 'manual/base')
print('Base answer:', (base_run / 'answer.txt').read_text().strip())
print('Wrong: the initial script discarded -20.10.')

## 2. Call Harness² directly

`run_task()` is the one-task orchestration function. `modes=('seq',)` selects sequential recursion; `k=3` requests three Propose–Probe–Compose rounds. `DemoProposer` makes deterministic edits; a real experiment replaces it with model calls while using the same recursion code.

In [ ]:
proposer = DemoProposer(adapter.spec.channels['proposer'])

result = run_task(
    task_id=task_id,
    k=3,
    adapter=adapter,
    prop=proposer,
    runs_root=adapter.spec.runs_root,
    modes=('seq',),
)

print('Status:', result.status)
print('Selected:', result.selections['seq']['chosen_label'])

## 3. See the recursive loop

Every round creates two candidates (`invN`, `domN`) and one composition (`cN`). Each `cN` becomes the starting harness for the next round.

In [ ]:
for number, call in enumerate(proposer._calls, 1):
    first_line = call.text.splitlines()[0] if call.text else ''
    print(f'{number:2}. {call.label:24} ok={call.ok}  {first_line}')

In [ ]:
mode = result.modes['seq']
print('Stages from run_sequential():')
for stage in mode['stages']:
    print(f"  {stage['label']:5} {stage['dim']}")
print('Final compositions:', mode['finals'])

Expected structure:

```text
round 1: inv1 + dom1 -> c1
round 2: inv2 + dom2 -> c2   (starts from c1)
round 3: inv3 + dom3 -> c3   (starts from c2)
```

In [ ]:
print('All probe/composition answers:')
for answer_file in sorted(OUTPUT.rglob('answer.txt')):
    relative = answer_file.relative_to(OUTPUT)
    print(f'{str(relative):80} {answer_file.read_text().strip()}')

## 4. Inspect how the harness changed

Harnesses are ordinary directories. The base contains `if amount > 0`; improved candidates remove it.

In [ ]:
scripts = sorted(adapter.spec.runs_root.rglob('harnesses/*/scripts/summarize.py'))
for script in scripts:
    source = script.read_text()
    state = 'BUGGY' if 'if amount > 0' in source else 'FIXED'
    label = script.parents[1].name
    print(f'{label:8} {state:5}  {script.relative_to(OUTPUT)}')

chosen = result.selections['seq']['chosen_label']
chosen_script = [p for p in scripts if p.parents[1].name == chosen][-1]
print('\nSELECTED SCRIPT:\n')
print(chosen_script.read_text())

## 5. Read the actual implementation loop

This prints the installed `run_sequential()` source. Find `for t in range(1, k + 1)`, `_run_stage`, `_roll_many`, and `current_harness = comp.tree`.

In [ ]:
import inspect
from harness2.core.recursion import run_sequential
print(inspect.getsource(run_sequential))

## The second recursion level

Within one task, `run_sequential()` carries `c1 → c2 → c3`. Across tasks, `run_domain()` calls `run_task()` repeatedly and stores the selected harness plus evidence in a library; the next task receives that library.

Parallel mode uses `modes=('agg',)`: it produces `2k` candidates from the same base and `k` independent compositions, so it explores wider rather than chaining compositions.

## Real benchmark

The CLI constructs the real adapter, proposer, selector, credentials, and model endpoints:

```bash
harness2-setup jb
harness2-check --bench jb
harness2 --bench jb --k 1 --mode sequential --max-tasks 1 --dry-run
harness2 --bench jb --k 1 --mode sequential --max-tasks 1
```

Real runs cost API money. See `docs/SETUP.md` and `docs/EXPERIMENTS.md` first.